# **1. GloVe Implementation from Scratch**

In [ ]:
import numpy as np
from collections import defaultdict
import re

with open("glove_corpus.txt", "r") as f:
    text = f.read().lower()

tokens = re.findall(r'\b\w+\b', text)
vocab = list(set(tokens))
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
vocab_size = len(vocab)

window_size = 2
cooc_matrix = defaultdict(float)

for i, target_word in enumerate(tokens):
    target_id = word_to_id[target_word]
    start = max(0, i - window_size)
    end = min(len(tokens), i + window_size + 1)

    for j in range(start, end):
        if i != j:
            context_id = word_to_id[tokens[j]]
            distance = abs(i - j)
            cooc_matrix[(target_id, context_id)] += 1.0 / distance

vector_size = 10
np.random.seed(42)
W = (np.random.rand(vocab_size, vector_size) - 0.5) / float(vector_size + 1)
W_tilde = (np.random.rand(vocab_size, vector_size) - 0.5) / float(vector_size + 1)
b = (np.random.rand(vocab_size) - 0.5) / float(vector_size + 1)
b_tilde = (np.random.rand(vocab_size) - 0.5) / float(vector_size + 1)

x_max = 100
alpha = 0.75
learning_rate = 0.05
epochs = 500

for epoch in range(epochs):
    global_cost = 0
    for (w_i, w_j), count in cooc_matrix.items():
        weight = (count / x_max) ** alpha if count < x_max else 1.0

        inner_cost = np.dot(W[w_i], W_tilde[w_j]) + b[w_i] + b_tilde[w_j] - np.log(count)
        cost = weight * (inner_cost ** 2)
        global_cost += 0.5 * cost

        grad_main = weight * inner_cost
        grad_W = grad_main * W_tilde[w_j]
        grad_W_tilde = grad_main * W[w_i]

        W[w_i] -= learning_rate * grad_W
        W_tilde[w_j] -= learning_rate * grad_W_tilde
        b[w_i] -= learning_rate * grad_main
        b_tilde[w_j] -= learning_rate * grad_main

word_vectors = W + W_tilde

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

target = "king"
if target in word_to_id:
    target_vec = word_vectors[word_to_id[target]]
    similarities = {}
    for word, idx in word_to_id.items():
        if word != target:
            similarities[word] = cosine_similarity(target_vec, word_vectors[idx])

    sorted_similar = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    print(f"Most similar words to '{target}' (Scratch Implementation):")
    for word, sim in sorted_similar[:5]:
        print(f"{word}: {sim:.4f}")

Most similar words to 'king' (Scratch Implementation):
queen: 0.9909
the: 0.9824
cat: 0.9779
and: 0.9621
rules: 0.9605


# **2. GloVe Implementation using Pretrained Model (Gensim)**

In [ ]:
!pip install gensim
import gensim.downloader as api

print("Downloading/Loading pretrained GloVe model (glove-wiki-gigaword-50)...")
model = api.load("glove-wiki-gigaword-50")

target_word = "king"

if target_word in model:
    print(f"\nMost similar words to '{target_word}' (Pretrained Gensim Model):")
    similar_words = model.most_similar(target_word, topn=5)
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")
else:
    print(f"'{target_word}' not found in the vocabulary.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 107.4 MB/s  0:00:00
Downloading/Loading pretrained GloVe model (glove-wiki-gigaword-50)...
[==================================================] 100.0% 66.0/66.0MB downloaded

Most similar words to 'king' (Pretrained Gensim Model):
prince: 0.8236
queen: 0.7839
ii: 0.7746
emperor: 0.7736
son: 0.7667


# **3. N-gram Statistical Language Model**

In [ ]:
import re
from collections import defaultdict, Counter

corpus = (
    "natural language processing is a branch of artificial intelligence.natural "
    "language processing enables computers to understand human language.machine "
    "learning is a branch of artificial intelligence.deep learning is a part of "
    "machine learning.language models learn patterns from text data.a language "
    "model predicts the next word in a sentence.natural language processing uses "
    "language models.artificial intelligence includes machine learning and deep "
    "learning."
)

clean_corpus = corpus.replace(".", " . ").lower()
tokens = clean_corpus.split()

bigram_counts = defaultdict(Counter)
unigram_counts = Counter(tokens)

for i in range(len(tokens) - 1):
    current_word = tokens[i]
    next_word = tokens[i + 1]
    bigram_counts[current_word][next_word] += 1

bigram_probs = defaultdict(dict)
for current_word, next_word_counts in bigram_counts.items():
    total_occurrences = unigram_counts[current_word]
    for next_word, count in next_word_counts.items():
        bigram_probs[current_word][next_word] = count / total_occurrences

def predict_next_word(sequence):
    seq_tokens = sequence.lower().split()

    if not seq_tokens:
        return "Please provide a valid input word."

    last_word = seq_tokens[-1]

    if last_word in bigram_probs:
        next_word_predictions = bigram_probs[last_word]
        best_next_word = max(next_word_predictions, key=next_word_predictions.get)
        prob = next_word_predictions[best_next_word]
        return f"Predicted next word: '{best_next_word}' (Probability: {prob:.2f})"
    else:
        return f"Word '{last_word}' not found in the model's vocabulary."

test_phrases = ["natural language", "artificial", "machine"]
print("N-Gram Prediction Results:")
for phrase in test_phrases:
    print(f"Given: '{phrase}' -> {predict_next_word(phrase)}")

N-Gram Prediction Results:
Given: 'natural language' -> Predicted next word: 'processing' (Probability: 0.43)
Given: 'artificial' -> Predicted next word: 'intelligence' (Probability: 1.00)
Given: 'machine' -> Predicted next word: 'learning' (Probability: 1.00)
